# ASSET-2 stage 4: leakage scan (Gate B)
Script `scripts/asset2_stage4_leaks.py`, version 2026-10-12. CPU runtime is enough. No API call, no GPU, no cost.
The notebook prints ids and counts only. Files with question text go to Drive `english/stage4/`.
Clear all outputs before this notebook is saved to the repo.

In [ ]:
# Cell 0: Drive, repo, then one PASS or FAIL line per input
EN_SHA = ''   # empty = final file version 2 (99a67e08...). For version 3 paste its full sha256 here.
import os, re, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
REPO = '/content/banglaindoorvqa-main'
EN_ARG = f'--english-sha256 {EN_SHA}' if EN_SHA else ''
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/blkeyd/banglaindoorvqa-main.git'
steps = ([['git', '-C', REPO, 'pull', '--ff-only', '-q']] if os.path.isdir(REPO)
         else [['git', 'clone', '-q', url, REPO]])
steps.append(['git', '-C', REPO, 'fetch', '--tags', '-q'])
ok = True
for cmd in steps:
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        ok = False
        msg = (r.stderr or r.stdout).replace(token, '***')
        print(re.sub(r'https://[^@\s]+@', 'https://***@', msg)[-500:])
print(('PASS' if ok else 'FAIL') + '  repo clone or pull: ' + REPO)
os.chdir(REPO)
!python scripts/asset2_stage4_leaks.py check-inputs {EN_ARG}

In [ ]:
# Cell 1: self test on made-up data (reads no dataset file). Expect: 26 of 26 self checks passed
!python scripts/asset2_stage4_leaks.py selftest

In [ ]:
# Cell 2: the leak scan. Writes Drive english/stage4/<UTC time>_en-<first 8 hex of the file's sha256>/
!python scripts/asset2_stage4_leaks.py scan {EN_ARG}

In [ ]:
# Cell 3: only after the two post-editors filled the fix sheet (D218).
# Save the filled copy next to leak_fix_sheet.csv in the run folder, as CSV UTF-8 or .xlsx.
FIX = '/content/drive/MyDrive/BanglaIndoorVQA/english/stage4/<run folder>/leak_fix_sheet_filled.csv'
!python scripts/asset2_stage4_leaks.py check-fixes --fix-sheet "{FIX}" {EN_ARG}